# Benchmark: TokenVector.Inference vs PyTorch trên Colab

So sánh cùng một workload: **MLP 4→8→3 ReLU, 20 mẫu, cross-entropy, AdamW (warmup 5 + cosine), clip, accum=2, 400 bước**.
- `.tkv`: `demo/trainer_tests.dll` (đã PASS 15/15, loss cuối khớp PyTorch 1e-07)
- `PyTorch`: reimplemented inline trong notebook với cùng init weights, cùng data, cùng lịch lr, cùng sampler

Kết quả kỳ vọng: `eval_loss ≈ 0.0149499` cho cả hai, acc = 1.0.
Timing = wall-clock (includes process startup cho .tkv; bao gồm chỉ train loop cho torch).


In [ ]:
# [1/6] Cài .NET 10
!curl -fsSL https://dot.net/v1/dotnet-install.sh -o /tmp/dotnet-install.sh
!bash /tmp/dotnet-install.sh --channel 10.0 --runtime dotnet --install-dir /opt/dotnet
import os
os.environ["PATH"] = "/opt/dotnet:" + os.environ["PATH"]
!/opt/dotnet/dotnet --version


In [ ]:
# [2/6] Clone repo
!git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.Inference.git
%cd TokenVector.Inference


In [ ]:
# [3/6] Benchmark .tkv — 5 lần chạy, median
import subprocess, statistics, time

t_tkv_runs = []
out = None
for _ in range(5):
    t0 = time.perf_counter()
    out = subprocess.check_output(["/opt/dotnet/dotnet", "demo/trainer_tests.dll"], text=True)
    t_tkv_runs.append(time.perf_counter() - t0)
print('.tkv runs (s):', [round(t, 3) for t in t_tkv_runs])
print('.tkv median (s):', round(statistics.median(t_tkv_runs), 3))
for ln in out.splitlines()[-4:]:
    print(' ', ln)


In [ ]:
# [4/6] PyTorch reference (cùng init, cùng sampler LCG, cùng lịch lr)
import math, statistics, time
import torch
import torch.nn.functional as F

secs, cur = {}, None
for ln in open('tv/tests/fixtures/train_cases.txt', encoding='utf-8'):
    if ln.startswith('#sec '):
        cur = ln[5:].strip(); secs[cur] = []
    elif ln.startswith('#') or not ln.strip():
        continue
    else:
        if cur == 'params':
            secs[cur].append(ln.strip())
        else:
            secs[cur].append([float(x) for x in ln.split()])

params = {}
for tok in secs['params'][0].split():
    k, v = tok.split('=')
    params[k] = float(v)
inp, hid, out_d = int(params['in']), int(params['hid']), int(params['out'])
warmup, total = int(params['warmup']), int(params['total'])
lr_max, lr_min = params['lr_max'], params['lr_min']
wd, b1, b2, eps, clip, accum = params['wd'], params['b1'], params['b2'], params['eps'], params['clip'], int(params['accum'])

init = secs['init']
w1 = torch.tensor(init[0], dtype=torch.float32).reshape(hid, inp)
b1v = torch.tensor(init[1], dtype=torch.float32)
w2 = torch.tensor(init[2], dtype=torch.float32).reshape(out_d, hid)
b2v = torch.tensor(init[3], dtype=torch.float32)

xb = torch.tensor(secs['x'], dtype=torch.float32)
yb = torch.tensor(secs['y'][0], dtype=torch.long)
NS = xb.shape[0]

def rng_step(s):
    return (s * 1103515245 + 12345) % (1 << 31)

def torch_train():
    p1, p2 = w1.clone().requires_grad_(True), w2.clone().requires_grad_(True)
    bb1, bb2 = b1v.clone().requires_grad_(True), b2v.clone().requires_grad_(True)
    opt = torch.optim.AdamW([p1, bb1, p2, bb2], lr=lr_max, betas=(b1, b2), eps=eps, weight_decay=wd)
    half = NS // accum
    # LCG Fisher-Yates (giống fixture + .tkv Sampler)
    state = 42
    perm = list(range(NS))
    i = NS - 1
    while i > 0:
        state = rng_step(state)
        j = state % (i + 1)
        perm[i], perm[j] = perm[j], perm[i]
        i = i - 1
    draws = 0
    def next_index():
        nonlocal state, perm, draws
        if draws > 0 and draws % NS == 0:
            perm = list(range(NS))
            i = NS - 1
            while i > 0:
                state = rng_step(state)
                j = state % (i + 1)
                perm[i], perm[j] = perm[j], perm[i]
                i = i - 1
        idx = perm[draws % NS]
        draws += 1
        return idx
    for step in range(total):
        lr = lr_max * (step + 1) / warmup if step < warmup else lr_min + (lr_max - lr_min) * 0.5 * (1 + math.cos(math.pi * (step - warmup) / (total - warmup)))
        for g in opt.param_groups: g['lr'] = lr
        opt.zero_grad(set_to_none=True)
        for acc in range(accum):
            idx = next_index()
            h = torch.relu(xb[idx:idx+1] @ p1.T + bb1)
            logits = h @ p2.T + bb2
            (F.cross_entropy(logits, yb[idx:idx+1]) / accum).backward()
        torch.nn.utils.clip_grad_norm_([p1, bb1, p2, bb2], clip)
        opt.step()
    with torch.no_grad():
        h = torch.relu(xb @ p1.T + bb1)
        logits = h @ p2.T + bb2
        ev = F.cross_entropy(logits, yb).item()
        acc = (logits.argmax(-1) == yb).float().mean().item()
    return ev, acc
t0 = time.perf_counter()
ev, acc = torch_train()
t_torch = time.perf_counter() - t0
print('eval_loss =', ev, ' acc =', acc)


In [ ]:
# [5/6] Benchmark PyTorch — 5 lần chạy
t_torch_runs = []
for _ in range(5):
    t0 = time.perf_counter()
    ev, acc = torch_train()
    t_torch_runs.append(time.perf_counter() - t0)
print('torch runs (s):', [round(t, 3) for t in t_torch_runs])
print('torch median (s):', round(statistics.median(t_torch_runs), 3))
print('eval_loss =', ev, ' acc =', acc)


In [ ]:
# [6/6] Bảng so sánh
print('| benchmark | wall-clock (s) |')
print('|---|---:|')
print(f'| .tkv trainer (.NET 10, 1 sample/step) | {statistics.median(t_tkv_runs):.3f} |')
print(f'| PyTorch (cùng workload, 1 sample/step) | {statistics.median(t_torch_runs):.3f} |')
print()
print('Note: .tkv thời gian bao gồm: 400 bước training + fixture parse + gate/cross-check PyTorch 1e-07.')
print('torch thời gian chỉ là vòng 400 bước training + eval.')
